# 03 · Controlled retrieval experiments

Investigate missing supporting sections before adding answer generation. The model is loaded from the existing ignored cache, and all computation stays local.
The original saved baseline is read, never overwritten. Outputs go to `outputs/retrieval_experiments/`.

| Run | Corpus | Ranking | Change from original dense |
| --- | --- | --- | --- |
| dense_original | Reviewed 512/64 LlamaIndex chunks | MiniLM cosine, max per section | Reproduction |
| dense_fitted | Model-token-budget chunks | Same model/cosine/aggregation | Chunking policy only |
| bm25_original | Original chunks | BM25, max per section | Scoring method only |
| rrf_original | Original chunks | Original dense + original BM25 section ranks | Fusion only |

The single fitted-chunk policy, BM25 settings, and fusion constant below are fixed before
examining results. No parameter sweep or repeated tuning is performed. The 20 questions are
a small development benchmark, not an independent test set. 

In [ ]:
import hashlib
import json
import os
import platform
import re
from collections import Counter
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "outputs/retrieval_baseline/results.json").is_file()
os.environ["HF_HOME"] = str(ROOT / ".cache/huggingface")
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

In [ ]:
import numpy as np
import torch
from llama_index.core.schema import TextNode
from sentence_transformers import SentenceTransformer

from fishrag.evaluation import gold_sections, metrics_at_k, rank_sections
from fishrag.ingestion import chunk_documents, load_documents

BASE = ROOT / "outputs/retrieval_baseline"
OUTPUT = ROOT / "outputs/retrieval_experiments"
OUTPUT.mkdir(parents=True, exist_ok=True)
MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_REVISION = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"
KS = (1, 3, 5, 10)
ORIGINAL_SIZE, ORIGINAL_OVERLAP = 512, 64
MAX_INPUT_TOKENS = 256  # Includes model special tokens, heading, and parent context.
BODY_OVERLAP_CAP = 32  # At most one quarter of a body's current window.
BM25_K1, BM25_B = 1.5, 0.75
RRF_CONSTANT = 60  # Equal weights; full section lists, no candidate cutoff.
DEVICE, BATCH_SIZE = "cpu", 16
torch.manual_seed(0)
torch.set_num_threads(4)


def file_hash(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


baseline_files = [
    BASE / name
    for name in ("results.json", "rankings.json", "corpus.json", "embeddings.npz")
]
baseline_hashes = {
    str(path.relative_to(ROOT)): file_hash(path) for path in baseline_files
}
saved = json.loads((BASE / "results.json").read_text(encoding="utf-8"))
saved_rankings = json.loads((BASE / "rankings.json").read_text(encoding="utf-8"))
manifest = json.loads((ROOT / "data/sources.json").read_text(encoding="utf-8"))
benchmark = json.loads((ROOT / "evals/questions.json").read_text(encoding="utf-8"))
questions = benchmark["questions"]
for name in ("data/sources.json", "evals/questions.json", "src/fishrag/ingestion.py"):
    assert file_hash(ROOT / name) == saved["settings"]["input_file_sha256"][name], name
assert saved["settings"]["model_id"] == MODEL_ID
assert saved["settings"]["model_revision"] == MODEL_REVISION
assert saved["settings"]["ks"] == list(KS)
assert saved["settings"]["chunk_size"] == ORIGINAL_SIZE
assert saved["settings"]["chunk_overlap"] == ORIGINAL_OVERLAP
assert (
    saved["settings"]["source_snapshot_sha256"] == benchmark["source_snapshot_sha256"]
)

documents = load_documents(ROOT, manifest)
original = chunk_documents(documents, ORIGINAL_SIZE, ORIGINAL_OVERLAP)
old_corpus = json.loads((BASE / "corpus.json").read_text(encoding="utf-8"))
assert [{"text": node.text, "metadata": node.metadata} for node in original] == [
    {"text": node["text"], "metadata": node["metadata"]} for node in old_corpus
]
assert len({q["id"] for q in questions}) == len(questions) == 20
assert sum(q["requires_multiple_sections"] for q in questions) == 9

## 1. Reproduce the original dense baseline

The shared loader and splitter are unchanged. `fishrag.evaluation` contains the same
gold-section, deduplication, and metric functions used by notebook 02. We re-encode both
chunks and original question text, then compare vectors, full section ordering, and every
saved per-question metric. A changed corpus or label file stops the experiment.
Metadata is retained alongside vectors, but embedding inputs contain only chunk/question text.

In [ ]:
model = SentenceTransformer(
    MODEL_ID,
    revision=MODEL_REVISION,
    device=DEVICE,
    cache_folder=str(ROOT / ".cache/huggingface/hub"),
    local_files_only=True,
    trust_remote_code=False,
    model_kwargs={"use_safetensors": True},
)
model.eval()
assert next(model.parameters()).dtype == torch.float32
assert model.max_seq_length == MAX_INPUT_TOKENS
tokenizer = model.tokenizer
assert tokenizer.is_fast, "Exact source character offsets need the fast tokenizer."


def token_count(text):
    return len(
        tokenizer(text, add_special_tokens=True, truncation=False, verbose=False)[
            "input_ids"
        ]
    )


def encode(texts):
    vectors = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
        convert_to_numpy=True,
        precision="float32",
        show_progress_bar=False,
    )
    assert np.isfinite(vectors).all()
    assert np.allclose(np.linalg.norm(vectors, axis=1), 1, atol=1e-5)
    return vectors


def section_keys(nodes):
    return [(node.metadata["source_id"], node.metadata["section"]) for node in nodes]


query_texts = [q["question"] for q in questions]
assert all(token_count(text) <= MAX_INPUT_TOKENS for text in query_texts)
query_vectors = encode(query_texts)
original_vectors = encode([node.text for node in original])
original_scores = query_vectors @ original_vectors.T
original_ranks = [rank_sections(row, section_keys(original)) for row in original_scores]
with np.load(BASE / "embeddings.npz") as old_vectors:
    vector_drift = {
        "chunks_max_abs": float(
            np.max(np.abs(original_vectors - old_vectors["chunks"]))
        ),
        "queries_max_abs": float(
            np.max(np.abs(query_vectors - old_vectors["questions"]))
        ),
    }
    assert np.allclose(original_vectors, old_vectors["chunks"], atol=1e-5)
    assert np.allclose(query_vectors, old_vectors["questions"], atol=1e-5)
for q, ranked in zip(questions, original_ranks, strict=True):
    assert [row["key"] for row in ranked] == [
        (row["source_id"], row["section"]) for row in saved_rankings[q["id"]]
    ]
original_truncated = [
    i for i, node in enumerate(original) if token_count(node.text) > MAX_INPUT_TOKENS
]
print("Baseline reproduced:", vector_drift)
print("Original chunks:", len(original), "Truncated inputs:", len(original_truncated))

## 2. Model-aware smaller chunks

A LlamaIndex token is not necessarily a MiniLM WordPiece token. Count the actual input,
including `[CLS]` and `[SEP]`. Each fitted chunk repeats its full section heading and any
parent regulatory introduction already present in the reviewed document. The body is sliced
using original character offsets, so Unicode and source text are not rewritten by decoding tokens.
Body windows overlap by at most 32 tokens, capped at a quarter of that window to make progress.

The usable body budget depends on prefix length. Some non-angling introductions nearly fill
the model limit, yielding very short body fragments. We preserve them and report that cost;
we do not silently drop context. This experiment changes the **chunking policy** (including
repeated prefixes), not just a nominal length. More chunks also give max-score aggregation
more opportunities for a high score; section Recall alone cannot isolate that effect.

In [ ]:
def fitted_chunks(documents):
    nodes, bodies, audit = [], {}, []
    for doc in documents:
        meta = doc.metadata
        key = (meta["source_id"], meta["section"])
        parents = (
            meta["parent_context_sections"].split(" | ")
            if meta["parent_context_sections"]
            else []
        )
        prefix = meta["section"] + "\n\n"
        if parents:
            prefix += (
                "\n\n".join(
                    f"Parent section: {heading}\n{bodies[(key[0], heading)]}"
                    for heading in parents
                )
                + "\n\n"
            )
        assert doc.text.startswith(prefix), key
        body = doc.text[len(prefix) :]
        bodies[key] = body
        capacity = MAX_INPUT_TOKENS - token_count(prefix)
        assert capacity > 0, (key, "Prefix leaves no body budget")
        offsets = tokenizer(
            body,
            add_special_tokens=False,
            truncation=False,
            return_offsets_mapping=True,
            verbose=False,
        )["offset_mapping"]
        assert offsets, key
        start, covered_until, windows = 0, 0, 0
        while start < len(offsets):
            end = min(start + capacity, len(offsets))
            char_start = 0 if start == 0 else offsets[start][0]
            while end > start:
                char_end = offsets[end][0] if end < len(offsets) else len(body)
                text = prefix + body[char_start:char_end]
                if token_count(text) <= MAX_INPUT_TOKENS:
                    break
                end -= 1
            assert end > start, (key, "No body token fits")
            assert char_start <= covered_until  # No omitted source characters.
            covered_until = max(covered_until, char_end)
            metadata = dict(meta)
            metadata.update(body_char_start=char_start, body_char_end=char_end)
            nodes.append(
                TextNode(
                    id_=f"{doc.id_}:fitted:{windows}",
                    text=text,
                    metadata=metadata,
                    excluded_embed_metadata_keys=list(metadata),
                    excluded_llm_metadata_keys=list(metadata),
                )
            )
            audit.append(
                {
                    "key": key,
                    "prefix_tokens": token_count(prefix),
                    "body_tokens": end - start,
                    "input_tokens": token_count(text),
                }
            )
            windows += 1
            if end == len(offsets):
                break
            overlap = min(BODY_OVERLAP_CAP, (end - start) // 4)
            start = end - overlap
        assert covered_until == len(body), key
    return nodes, audit


fitted, chunk_audit = fitted_chunks(documents)
assert set(section_keys(fitted)) == set(section_keys(original))
parent_lookup = {(d.metadata["source_id"], d.metadata["section"]): d for d in documents}
for node in fitted:
    parent = parent_lookup[(node.metadata["source_id"], node.metadata["section"])]
    assert all(node.metadata[key] == value for key, value in parent.metadata.items())
    assert node.text.startswith(parent.metadata["section"] + "\n\n")
    full_ids = tokenizer(node.text, truncation=False, verbose=False)["input_ids"]
    limited_ids = tokenizer(node.text, truncation=True, max_length=MAX_INPUT_TOKENS)[
        "input_ids"
    ]
    actual_ids = model.tokenize([node.text])["input_ids"][0].tolist()
    assert full_ids == limited_ids == actual_ids, "Silent model truncation detected"
    assert len(full_ids) <= MAX_INPUT_TOKENS
fitted_vectors = encode([node.text for node in fitted])
fitted_scores = query_vectors @ fitted_vectors.T
fitted_ranks = [rank_sections(row, section_keys(fitted)) for row in fitted_scores]
print(
    {
        "fitted_chunks": len(fitted),
        "max_input_tokens": max(row["input_tokens"] for row in chunk_audit),
        "min_body_tokens": min(row["body_tokens"] for row in chunk_audit),
        "tiny_body_windows_at_most_8_tokens": sum(
            row["body_tokens"] <= 8 for row in chunk_audit
        ),
        "truncated_inputs": 0,
    }
)

## 3. BM25 on the original chunks

BM25 rewards exact query-word matches, discounts common corpus terms, saturates repeated
term frequency, and normalizes for chunk length. This transparent implementation uses
`k1=1.5`, `b=0.75`, and positive IDF `log(1 + (N - df + 0.5)/(df + 0.5))`.
Tokenization is lowercase ASCII letter/digit runs, with no stemming or stop-word filtering.
Each query term contributes once. These settings were fixed without searching the benchmark.
Scores are computed on the **original chunks**, then collapsed by maximum score per section.
Zero-score ties use corpus order; they are arbitrary ordering, not evidence of relevance.

In [ ]:
def words(text):
    return re.findall(r"[a-z0-9]+", text.casefold())


def bm25_scores(texts, queries, k1=BM25_K1, b=BM25_B):
    counts = [Counter(words(text)) for text in texts]
    lengths = np.array([sum(count.values()) for count in counts], dtype=float)
    avg_length = float(lengths.mean())
    assert avg_length > 0
    df = Counter(term for count in counts for term in count)
    normalizer = k1 * (1 - b + b * lengths / avg_length)
    result = np.zeros((len(queries), len(texts)), dtype=float)
    for i, query in enumerate(queries):
        for term in sorted(set(words(query))):
            idf = np.log1p((len(texts) - df[term] + 0.5) / (df[term] + 0.5))
            frequency = np.array([count[term] for count in counts], dtype=float)
            result[i] += idf * frequency * (k1 + 1) / (frequency + normalizer)
    return result


# Hand-check a one-term hit and a missing term before using corpus results.
fixture = bm25_scores(["bass bass", "trout trout"], ["bass", "unseen"])
assert np.isclose(fixture[0, 0], np.log(2) * 2 * 2.5 / 3.5)
assert fixture[0, 1] == 0 and np.all(fixture[1] == 0)
bm25_matrix = bm25_scores([node.text for node in original], query_texts)
bm25_ranks = [rank_sections(row, section_keys(original)) for row in bm25_matrix]

## 4. Reciprocal rank fusion of original dense and BM25

RRF uses ranks rather than incomparable cosine/BM25 score scales:
`fused_score(section) = 1/(60 + dense_rank) + 1/(60 + bm25_rank)`.
Both inputs are already deduplicated section lists. Use equal weights and their full 150-section
lists. Stable ties follow corpus section order. This run uses **original** dense ranks, not the
new fitted chunks, so fusion is not confounded with the chunking comparison.

In [ ]:
section_order = list(dict.fromkeys(section_keys(original)))


def fuse(dense, lexical, constant=RRF_CONSTANT):
    totals = dict.fromkeys(section_order, 0.0)
    for ranking in (dense, lexical):
        for rank, row in enumerate(ranking, 1):
            totals[row["key"]] += 1 / (constant + rank)
    ordered = sorted(section_order, key=lambda key: -totals[key])
    return [{"key": key, "score": totals[key]} for key in ordered]


rrf_ranks = [
    fuse(dense, lexical)
    for dense, lexical in zip(original_ranks, bm25_ranks, strict=True)
]
assert np.isclose(fuse(original_ranks[0], original_ranks[0])[0]["score"], 2 / 61)
rankings = {
    "dense_original": original_ranks,
    "dense_fitted": fitted_ranks,
    "bm25_original": bm25_ranks,
    "rrf_original": rrf_ranks,
}
for method in rankings.values():
    for ranked in method:
        assert len(ranked) == len(section_order)
        assert len({row["key"] for row in ranked}) == len(section_order)

## 5. Fixed section-level evaluation

Recall@k is the fraction of required sections retrieved among the first k unique sections.
MRR@k averages `1 / first relevant rank`, or zero if none occurs by k. All-required coverage
is reported separately for the nine multi-section questions, since a first hit is insufficient.
We reuse notebook 02's metric functions and test known deduplication and rank cases.

Report macro averages for 17 supported questions, one partially supported question's labeled
general guidance, and their combined 18 questions. The original partial query is embedded as
written; only its labeled supported section is scored. Its missing lake-specific information
remains unsupported. The two wholly unsupported questions get diagnostic rankings only:
no relevance labels, metric denominators, or answerability thresholds are invented.

In [ ]:
a, b = ("source", "A"), ("source", "B")
test_ranks = rank_sections([0.9, 0.8, 0.7], [a, a, b])
assert [row["key"] for row in test_ranks] == [a, b]
assert len(rank_sections([1, 1], [a, ("other", "A")])) == 2
assert metrics_at_k(test_ranks, {a, b}, 1) == {
    "recall": 0.5,
    "reciprocal_rank": 1.0,
    "all_required_found": False,
}
assert metrics_at_k(test_ranks, {b}, 1)["reciprocal_rank"] == 0
assert metrics_at_k(test_ranks, {b}, 3)["reciprocal_rank"] == 0.5

per_question, summary = {}, []
for name, method in rankings.items():
    rows = []
    for question, ranked in zip(questions, method, strict=True):
        gold = gold_sections(question)
        assert gold <= set(section_order)
        if question["answerability"] == "unsupported":
            assert not gold
        else:
            assert gold
        rows.append(
            {
                "id": question["id"],
                "answerability": question["answerability"],
                "requires_multiple_sections": question["requires_multiple_sections"],
                "metrics": {str(k): metrics_at_k(ranked, gold, k) for k in KS}
                if gold
                else None,
                "required_section_ranks": [
                    {
                        "source_id": key[0],
                        "section": key[1],
                        "rank": next(
                            i for i, row in enumerate(ranked, 1) if row["key"] == key
                        ),
                    }
                    for key in sorted(gold)
                ],
            }
        )
    per_question[name] = rows
    groups = {
        "supported": [row for row in rows if row["answerability"] == "supported"],
        "partial_supported_portion": [
            row for row in rows if row["answerability"] == "partially_supported"
        ],
        "all_labeled": [row for row in rows if row["metrics"] is not None],
    }
    for group_name, group in groups.items():
        multi = [row for row in group if row["requires_multiple_sections"]]
        for k in KS:
            summary.append(
                {
                    "method": name,
                    "group": group_name,
                    "k": k,
                    "questions": len(group),
                    "recall": float(
                        np.mean([row["metrics"][str(k)]["recall"] for row in group])
                    ),
                    "mrr": float(
                        np.mean(
                            [row["metrics"][str(k)]["reciprocal_rank"] for row in group]
                        )
                    ),
                    "multi_section_questions": len(multi),
                    "all_required_found_count": sum(
                        row["metrics"][str(k)]["all_required_found"] for row in multi
                    ),
                    "all_required_found_rate": float(
                        np.mean(
                            [
                                row["metrics"][str(k)]["all_required_found"]
                                for row in multi
                            ]
                        )
                    )
                    if multi
                    else None,
                }
            )

for current, old in zip(
    per_question["dense_original"], saved["per_question"], strict=True
):
    assert current["id"] == old["id"] and current["metrics"] == old["metrics"]
    assert current["required_section_ranks"] == old["required_section_ranks"]
for row in summary:
    if row["method"] == "dense_original":
        old = next(
            s
            for s in saved["summary"]
            if s["group"] == row["group"] and s["k"] == row["k"]
        )
        assert all(row[key] == value for key, value in old.items())
    print(json.dumps(row))

## 6. Inspect gains, regressions, and encoded facts

Compare each required section's rank for Scugog, Chemong, aggregate limits, livewell definitions,
and the partly supported depth question. Print every changed Recall@k/all-required flag, not
just wins. Section hits do not guarantee complete fact coverage in the winning chunk.
For the two previously truncated sections, inspect original encoded tails and representative
new fragments containing the formerly omitted crayfish preference and Trent River season.

In [ ]:
for index in (8, 11, 12, 13, 14, 17):
    question = questions[index]
    print("\n", question["id"], question["question"])
    for name, rows in per_question.items():
        print(name, rows[index]["required_section_ranks"])

changes = []
for name, rows in per_question.items():
    if name == "dense_original":
        continue
    for row, baseline in zip(rows, per_question["dense_original"], strict=True):
        if row["metrics"] is None:
            continue
        for k in KS:
            new, old = row["metrics"][str(k)], baseline["metrics"][str(k)]
            if new != old:
                changes.append(
                    {
                        "method": name,
                        "id": row["id"],
                        "k": k,
                        "before": old,
                        "after": new,
                    }
                )
print("\nAll per-question metric changes:")
for row in changes:
    print(json.dumps(row))

fact_probes = [
    (
        "ontario-learn-to-fish",
        (
            "Identifying, catching and handling fish species > "
            "Smallmouth and largemouth bass"
        ),
        "crayfish",
    ),
    ("ontario-zone-17", "Waterbody exceptions", "Walleye - open from January 1"),
]
fact_checks = []
for source_id, heading, phrase in fact_probes:
    key = (source_id, heading)
    original_indices = [
        i for i, section in enumerate(section_keys(original)) if section == key
    ]
    encoded_tails = [
        tokenizer.decode(
            tokenizer(original[i].text, truncation=True, max_length=MAX_INPUT_TOKENS)[
                "input_ids"
            ]
        )[-450:]
        for i in original_indices
    ]
    candidates = [
        i
        for i, node in enumerate(fitted)
        if section_keys([node])[0] == key and phrase in node.text
    ]
    assert candidates, (key, phrase)
    print("\nFact probe:", phrase, key, "Original encoded tails:", encoded_tails)
    for i in candidates:
        print(
            "Fitted fragment", i, "tokens:", token_count(fitted[i].text), fitted[i].text
        )
    fact_checks.append(
        {
            "source_id": source_id,
            "section": heading,
            "phrase": phrase,
            "fitted_chunk_indices": candidates,
        }
    )

print("\nUnsupported diagnostic rankings (scores are not answerability):")
for i, question in enumerate(questions):
    if question["answerability"] == "unsupported":
        print(question["id"], question["missing_evidence"])
        for name, method in rankings.items():
            print(name, method[i][:3])

## 7. Save the experiment, not the original baseline

Save actual metrics, all section ranks, provenance-bearing chunk corpora, token budgets,
vectors, settings, package versions, and hashes under the ignored experiment directory.
The source manifest and benchmark labels are not changed. No generated source text or saved
output belongs in the tracked notebook. The original saved baseline remains byte-for-byte intact.

In [ ]:
settings = {
    "executed_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "model_max_seq_length": model.max_seq_length,
    "device": DEVICE,
    "batch_size": BATCH_SIZE,
    "torch_threads": torch.get_num_threads(),
    "precision": "float32",
    "normalized_embeddings": True,
    "prompt": None,
    "original_chunk_size": ORIGINAL_SIZE,
    "original_chunk_overlap": ORIGINAL_OVERLAP,
    "fitted_max_input_tokens": MAX_INPUT_TOKENS,
    "fitted_body_overlap_cap": BODY_OVERLAP_CAP,
    "fitted_overlap_rule": "min(32, floor(current_window_body_tokens / 4))",
    "fitted_prefix": "repeat full section heading and parent regulation context",
    "original_truncated_indices": original_truncated,
    "fitted_truncated_inputs": 0,
    "fitted_max_observed_tokens": max(row["input_tokens"] for row in chunk_audit),
    "bm25": {
        "k1": BM25_K1,
        "b": BM25_B,
        "idf": "log1p((N-df+0.5)/(df+0.5))",
        "tokens": "casefold then [a-z0-9]+; unique query terms; no stemming/stopwords",
    },
    "rrf": {
        "constant": RRF_CONSTANT,
        "weights": [1, 1],
        "candidate_cutoff": None,
        "inputs": ["dense_original", "bm25_original"],
    },
    "ks": list(KS),
    "section_score": "maximum chunk score; RRF fuses section ranks",
    "tie_break": "corpus order",
    "vector_drift_from_saved": vector_drift,
    "source_snapshot_sha256": benchmark["source_snapshot_sha256"],
    "saved_baseline_file_sha256": baseline_hashes,
    "input_file_sha256": {
        name: file_hash(ROOT / name)
        for name in (
            "data/sources.json",
            "evals/questions.json",
            "src/fishrag/ingestion.py",
            "src/fishrag/evaluation.py",
            "notebooks/01_document_ingestion.ipynb",
            "notebooks/02_retrieval_baseline.ipynb",
            "notebooks/03_retrieval_experiments.ipynb",
            "pyproject.toml",
        )
    },
    "python": platform.python_version(),
    "packages": {
        name: version(name)
        for name in (
            "sentence-transformers",
            "torch",
            "transformers",
            "numpy",
            "llama-index-core",
            "beautifulsoup4",
            "huggingface-hub",
        )
    },
    "corpus_sizes": {
        "original_chunks": len(original),
        "fitted_chunks": len(fitted),
        "unique_sections": len(section_order),
    },
}
for name, old_version in saved["settings"]["packages"].items():
    assert settings["packages"][name] == old_version, (name, "Package changed")
report = {
    "settings": settings,
    "summary": summary,
    "per_question": per_question,
    "changes_from_dense_original": changes,
    "fact_probes": fact_checks,
}
(OUTPUT / "results.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
(OUTPUT / "rankings.json").write_text(
    json.dumps(
        {
            name: {
                question["id"]: [{**row, "key": list(row["key"])} for row in ranked]
                for question, ranked in zip(questions, method, strict=True)
            }
            for name, method in rankings.items()
        },
        indent=2,
    ),
    encoding="utf-8",
)
(OUTPUT / "fitted_corpus.json").write_text(
    json.dumps(
        [
            {
                "chunk_index": i,
                "text": node.text,
                "metadata": node.metadata,
                "budget": chunk_audit[i],
            }
            for i, node in enumerate(fitted)
        ],
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)
np.savez(
    OUTPUT / "embeddings.npz",
    questions=query_vectors,
    original=original_vectors,
    fitted=fitted_vectors,
)
assert all(file_hash(ROOT / path) == digest for path, digest in baseline_hashes.items())
print("Saved experiment:", OUTPUT)

## Limits on choosing a configuration

Use the displayed results and regressions, not a single best score. Fitted chunks avoid
silent loss of text but can be fragmented and repetitive. BM25 can find exact terminology
but miss paraphrases. RRF can combine strengths and can also demote a good dense hit.
None of these methods guarantees fetching every exception or definition. Even full annotated
section coverage is not an answer-faithfulness result or proof of current fishing conditions.
Any preference from these 20 development questions is provisional, not independent test performance.

## Observations from the fixed local run (November 8, 2026)

The saved baseline reproduced with **zero vector drift**, identical full section rankings,
and identical per-question/aggregate metrics. The following are development-benchmark results.

**Supported questions (17)**. Values in each cell follow k = 1, 3, 5, 10.

| Method | Recall@k | MRR@k | All required: multi-section questions |
| --- | --- | --- | --- |
| Original dense | 0.5686, 0.8725, 0.8725, 0.9314 | 0.8235, 0.9020, 0.9020, 0.9020 | 0/9, 4/9, 4/9, 6/9 |
| Fitted dense | 0.5686, 0.8725, 0.8725, 0.9020 | 0.8235, 0.9118, 0.9118, 0.9118 | 0/9, 4/9, 4/9, 5/9 |
| BM25 | 0.5392, 0.7745, 0.8333, 0.8529 | 0.7647, 0.8431, 0.8431, 0.8431 | 0/9, 4/9, 6/9, 6/9 |
| RRF | 0.5686, 0.8235, 0.9020, 0.9314 | 0.8235, 0.8725, 0.8873, 0.8873 | 0/9, 5/9, 5/9, 6/9 |

**Supported portion of the partial question (1)**. Values in each cell follow k = 1, 3, 5, 10.

| Method | Recall@k | MRR@k | All required: multi-section questions |
| --- | --- | --- | --- |
| Original dense | 1.0000, 1.0000, 1.0000, 1.0000 | 1.0000, 1.0000, 1.0000, 1.0000 | Not applicable |
| Fitted dense | 1.0000, 1.0000, 1.0000, 1.0000 | 1.0000, 1.0000, 1.0000, 1.0000 | Not applicable |
| BM25 | 0.0000, 1.0000, 1.0000, 1.0000 | 0.0000, 0.3333, 0.3333, 0.3333 | Not applicable |
| RRF | 1.0000, 1.0000, 1.0000, 1.0000 | 1.0000, 1.0000, 1.0000, 1.0000 | Not applicable |

**Combined labeled questions (18)**. Values in each cell follow k = 1, 3, 5, 10.

| Method | Recall@k | MRR@k | All required: multi-section questions |
| --- | --- | --- | --- |
| Original dense | 0.5926, 0.8796, 0.8796, 0.9352 | 0.8333, 0.9074, 0.9074, 0.9074 | 0/9, 4/9, 4/9, 6/9 |
| Fitted dense | 0.5926, 0.8796, 0.8796, 0.9074 | 0.8333, 0.9167, 0.9167, 0.9167 | 0/9, 4/9, 4/9, 5/9 |
| BM25 | 0.5093, 0.7870, 0.8426, 0.8611 | 0.7222, 0.8148, 0.8148, 0.8148 | 0/9, 4/9, 6/9, 6/9 |
| RRF | 0.5926, 0.8333, 0.9074, 0.9352 | 0.8333, 0.8796, 0.8935, 0.8935 | 0/9, 5/9, 5/9, 6/9 |

**What changed, and what did not:**

- The fitted policy creates 248 chunks instead of 152, all at most 256 tokens including
  special tokens, full section headings, and parent regulatory context. Original-body character
  coverage and actual model input IDs are checked. No query is truncated. Of the fitted body
  windows, 89 contain at most eight tokens; the smallest contains three. Long repeated parent
  context accounts for this fragmentation. This is a documented cost, not a second tuned policy.
- The previously omitted crayfish preference and final Trent River walleye season now occur
  in fully encoded fragments. That fixes input loss, not necessarily retrieval: Scugog's waterbody
  exception moves from rank 10 to 11, reducing complete multi-section coverage at k=10 from
  6/9 to 5/9. Fish-handling's required section improves from rank 3 to 2.
- BM25 puts Scugog's waterbody exception first and its zone-wide rule fourth; it also moves
  the aggregate-limit definition to rank 3. It loses the fish-handling section from the top ten,
  and puts the partial question's general bass guidance third instead of first.
- RRF puts Scugog's exception second and the aggregate-limit definition seventh. However,
  the standalone zone-wide introduction for `fishrag-008` falls from rank 9 to 19, losing a
  required label at k=10. Its qualification is repeated in the bass rule's text, so this is a
  strict annotation-coverage regression rather than proof that all qualification text vanished.
- All four methods rank Chemong's local sanctuary restriction first. None retrieves its
  required general sanctuary definition in the top ten (ranks 36, 37, 68, 49 respectively).
  The livewell definition also remains outside the top ten (102, 103, 45, 66).
- RRF raises combined Recall@5 from 0.8796 to 0.9074, but its MRR@10 is lower and its
  complete multi-section coverage at k=10 remains 6/9. Its wins do not remove the regressions.

**No overall replacement is selected.** Retain original dense as the comparison reference;
the fitted policy addresses truncation, and RRF offers useful complementary matches, but neither
dominates on the fixed labels.